# 채팅 파이프라인 개선 테스트

## 파이프라인 구조
1. **요청 분류**: 자기소개서 요청 vs 대화 + 문항 유형 분류
2. **스토리라인 설계**: 적합성 점수 기반 경험 역할 분배 + 서술 흐름 설계
3. **초안 생성**: 스토리라인 + few-shot 예시 기반 생성
4. **검수 & 보정**: 글자수 체크 + 나열식 패턴 체크

## 0. 환경 설정

In [1]:
import os
import sys
from dotenv import load_dotenv

# 프로젝트 루트를 path에 추가
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# .env 로드
load_dotenv(os.path.join(project_root, ".env"))

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# LLM 초기화
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
)

classification_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
)

print(f"LLM 초기화 완료: {llm.model_name}")

LLM 초기화 완료: gpt-4o-mini


## 0-1. 테스트용 더미 데이터

In [2]:
# === 프로젝트 정보 ===
company = "네이버"
recruit_notice = """[네이버 백엔드 개발자 채용공고]
- 주요업무: 대규모 트래픽 처리, API 설계 및 개발, 데이터 파이프라인 구축
- 자격요건: Java/Python 등 1개 이상 언어 능숙, REST API 설계 경험, RDBMS/NoSQL 경험
- 우대사항: MSA 경험, 대규모 트래픽 처리 경험, 오픈소스 기여 경험"""

# === 문항 정보 ===
question = "본인이 주도적으로 문제를 해결한 경험과 그 과정에서 배운 점을 기술해주세요."
max_length = 500

# === 경험 데이터 (최대 3개, 적합성 점수 포함) ===
experiences_with_scores = [
    {
        "title": "Cardify 프로젝트 - 결제 시스템 장애 해결",
        "experience_type": "동아리 활동",
        "format_type": "STAR",
        "situation": "Cardify 서비스에서 결제 API 응답 시간이 평균 3초로 급증하여 사용자 이탈률이 40% 증가한 상황이었습니다.",
        "task": "결제 API 응답 시간을 1초 이내로 줄이고, 서비스 안정성을 확보해야 했습니다.",
        "action": "DB 쿼리 프로파일링으로 N+1 문제를 발견하고, Eager Loading과 Redis 캐싱을 도입했습니다. 또한 Circuit Breaker 패턴을 적용하여 외부 PG사 장애 전파를 차단했습니다.",
        "result": "API 응답 시간을 3초에서 200ms로 93% 개선했고, 사용자 이탈률이 40%에서 12%로 감소했습니다.",
        "category": "기술적 전문성",
        "tags": "백엔드, 트러블슈팅, API연동",
        "similarity_score": 0.92,  # 적합성 점수
    },
    {
        "title": "해커톤 - 실시간 채팅 서비스 개발",
        "experience_type": "동아리 활동",
        "format_type": "STAR",
        "situation": "48시간 해커톤에서 4인 팀으로 실시간 채팅 서비스를 개발해야 했습니다.",
        "task": "WebSocket 기반 실시간 메시징과 채팅방 관리 기능을 담당했습니다.",
        "action": "FastAPI + WebSocket으로 실시간 통신을 구현하고, Redis Pub/Sub으로 서버 간 메시지 동기화를 처리했습니다.",
        "result": "48시간 내에 MVP를 완성하여 참가팀 30팀 중 3등을 수상했습니다.",
        "category": "기술적 전문성",
        "tags": "백엔드, API연동, 협업도구(Notion/Jira/Slack)",
        "similarity_score": 0.78,  # 적합성 점수
    },
    {
        "title": "인턴십 - 데이터 파이프라인 자동화",
        "experience_type": "인턴",
        "format_type": "PSI",
        "problem": "매일 수동으로 3시간씩 데이터를 수집하고 정제하는 반복 업무가 있었습니다.",
        "solution": "Airflow를 활용하여 데이터 수집-정제-적재 파이프라인을 자동화하고, Slack 알림으로 모니터링 체계를 구축했습니다.",
        "insight": "자동화를 통해 주 15시간의 업무 시간을 절약했고, 데이터 정합성도 향상되었습니다.",
        "category": "논리적 분석력",
        "tags": "데이터분석, 프로세스개선, 인프라/클라우드",
        "similarity_score": 0.65,  # 적합성 점수
    },
]

# 사용자 메시지
user_message = "자기소개서 작성해줘"

print("더미 데이터 로드 완료")
print(f"경험 {len(experiences_with_scores)}개, 글자수 제한 {max_length}자")

더미 데이터 로드 완료
경험 3개, 글자수 제한 500자


---
## 1단계: 요청 분류 + 문항 유형 분류

In [3]:
from enum import Enum


class QuestionType(str, Enum):
    """자기소개서 문항 유형"""
    MOTIVATION = "지원동기"       # 지원동기 및 입사 후 포부
    COMPETENCY = "직무역량"       # 직무 관련 경험과 역량
    COLLABORATION = "협업소통"    # 팀워크, 소통 경험
    PROBLEM_SOLVING = "문제해결"  # 어려움/문제 극복 경험
    GROWTH = "성장가치관"         # 성장과정, 강점/약점, 가치관
    OTHER = "기타"               # 위 유형에 해당하지 않는 경우


class RequestClassification(BaseModel):
    """요청 분류 결과"""
    is_cover_letter_request: bool = Field(
        description="자기소개서 작성/수정 요청이면 True, 단순 질문/피드백 요청이면 False"
    )
    question_type: QuestionType = Field(
        description="문항 유형 분류 (자기소개서 요청일 때만 의미 있음)"
    )


CLASSIFICATION_PROMPT = """사용자의 요청과 문항을 분석하세요.

## 사용자 요청
{user_message}

## 문항
{question}

## 지시사항
1. 사용자 요청이 자기소개서 작성/수정 요청인지 판단하세요.
   - 작성, 수정, 다시 써줘 등 → True
   - 피드백, 첨삭, 질문, 조언 등 → False
2. 문항을 다음 유형 중 하나로 분류하세요:
   - 지원동기: 지원 이유, 입사 후 목표/포부
   - 직무역량: 직무 관련 기술/경험/역량
   - 협업소통: 팀워크, 협업, 커뮤니케이션, 갈등 해결
   - 문제해결: 문제/어려움 극복, 도전, 실패 경험
   - 성장가치관: 성장과정, 가치관, 강점/약점, 인생관
   - 기타: 위에 해당하지 않는 경우"""


async def classify_request(user_message: str, question: str) -> RequestClassification:
    """요청 분류 + 문항 유형 분류"""
    structured_llm = classification_llm.with_structured_output(RequestClassification)
    result = await structured_llm.ainvoke(
        CLASSIFICATION_PROMPT.format(
            user_message=user_message,
            question=question,
        )
    )
    return result


# 테스트
classification = await classify_request(user_message, question)
print(f"자기소개서 요청: {classification.is_cover_letter_request}")
print(f"문항 유형: {classification.question_type}")

자기소개서 요청: True
문항 유형: QuestionType.PROBLEM_SOLVING


---
## 2단계: 스토리라인 설계 (적합성 점수 기반)

In [4]:
class ExperienceRole(str, Enum):
    """스토리라인에서의 경험 역할"""
    PRIMARY = "주 경험"     # 상세하게 서술
    SUPPORTING = "보조 경험"  # 간략하게 언급


class StorylineExperience(BaseModel):
    """스토리라인에 포함된 경험"""
    title: str = Field(description="경험 제목")
    role: ExperienceRole = Field(description="주 경험 / 보조 경험")
    key_facts: list[str] = Field(description="이 경험에서 사용할 핵심 사실 (1~3개)")
    how_to_mention: str = Field(description="이 경험을 어떻게 언급할지 한 줄 가이드")


class Storyline(BaseModel):
    """자기소개서 스토리라인 설계"""
    core_message: str = Field(description="자기소개서의 핵심 메시지 (한 문장)")
    opening: str = Field(description="도입부 방향 (어떤 장면/사실로 시작할지)")
    development: str = Field(description="전개부 방향 (주 경험을 어떻게 풀어갈지)")
    conclusion: str = Field(description="마무리 방향 (어떻게 직무/회사와 연결할지)")
    experiences: list[StorylineExperience] = Field(description="경험별 역할 및 사용 계획")


def assign_experience_roles(
    experiences_with_scores: list[dict],
) -> str:
    """
    적합성 점수 기반으로 경험 역할을 분배하고 텍스트로 변환
    - 가장 높은 점수 → 주 경험 (상세 서술)
    - 나머지 → 보조 경험 (간략 언급)
    """
    sorted_exps = sorted(
        experiences_with_scores,
        key=lambda x: x["similarity_score"],
        reverse=True,
    )

    lines = []
    for i, exp in enumerate(sorted_exps):
        role = "[주 경험 - 상세 서술]" if i == 0 else "[보조 경험 - 간략 언급]"
        score = exp["similarity_score"]

        # 포맷에 따라 내용 구성
        if exp.get("format_type") == "STAR":
            content = (
                f"  상황: {exp.get('situation', 'N/A')}\n"
                f"  과제: {exp.get('task', 'N/A')}\n"
                f"  행동: {exp.get('action', 'N/A')}\n"
                f"  결과: {exp.get('result', 'N/A')}"
            )
        elif exp.get("format_type") == "PSI":
            content = (
                f"  문제: {exp.get('problem', 'N/A')}\n"
                f"  해결: {exp.get('solution', 'N/A')}\n"
                f"  인사이트: {exp.get('insight', 'N/A')}"
            )
        else:
            content = f"  내용: {exp.get('content', 'N/A')}"

        lines.append(
            f"### {exp['title']} {role} (적합도: {score:.2f})\n"
            f"- 유형: {exp['experience_type']}\n"
            f"- 카테고리: {exp['category']}\n"
            f"{content}"
        )

    return "\n\n".join(lines)


STORYLINE_PROMPT = """제공된 경험 정보를 바탕으로 자기소개서 스토리라인을 설계하세요.

## 문항
{question}

## 글자수 제한
{max_length}자

## 회사/직무
{company}

## 경험 정보 (역할이 지정되어 있음)
{experiences}

## 지시사항
1. [주 경험]을 중심으로 스토리를 설계하세요. 주 경험은 구체적 사실과 함께 상세히 서술합니다.
2. [보조 경험]은 주 경험을 뒷받침하거나 연결하는 용도로만 간략히 언급합니다. 보조 경험을 별도 단락으로 나열하지 마세요.
3. 경험을 나열하지 말고, 하나의 이야기 흐름으로 엮으세요.
4. 경험 정보에 없는 사실을 만들어내지 마세요.
5. 글자수 제한을 고려하여 현실적인 분량으로 설계하세요."""


async def design_storyline(
    question: str,
    max_length: int,
    company: str,
    experiences_with_scores: list[dict],
) -> Storyline:
    """스토리라인 설계"""
    experiences_text = assign_experience_roles(experiences_with_scores)
    structured_llm = classification_llm.with_structured_output(Storyline)
    result = await structured_llm.ainvoke(
        STORYLINE_PROMPT.format(
            question=question,
            max_length=max_length,
            company=company,
            experiences=experiences_text,
        )
    )
    return result


# 테스트
storyline = await design_storyline(question, max_length, company, experiences_with_scores)
print(f"핵심 메시지: {storyline.core_message}")
print(f"도입: {storyline.opening}")
print(f"전개: {storyline.development}")
print(f"마무리: {storyline.conclusion}")
print("---")
for exp in storyline.experiences:
    print(f"[{exp.role}] {exp.title}")
    print(f"  사용할 사실: {exp.key_facts}")
    print(f"  언급 방식: {exp.how_to_mention}")

핵심 메시지: 문제를 해결하는 과정에서 기술적 전문성과 팀워크의 중요성을 배웠습니다.
도입: 대학 동아리에서 진행한 Cardify 프로젝트는 결제 시스템의 장애로 인해 사용자 이탈률이 40%까지 증가한 위기 상황이었습니다. 이 문제를 해결하기 위해 팀원들과 함께 머리를 맞대고 해결책을 모색했습니다.
전개: 우선, DB 쿼리 프로파일링을 통해 N+1 문제를 발견했습니다. 이를 해결하기 위해 Eager Loading과 Redis 캐싱을 도입하여 데이터 처리 속도를 개선했습니다. 또한, Circuit Breaker 패턴을 적용하여 외부 PG사 장애가 발생하더라도 서비스가 안정적으로 운영될 수 있도록 했습니다. 이러한 기술적 접근을 통해 API 응답 시간을 3초에서 200ms로 단축시킬 수 있었습니다.
마무리: 결과적으로 사용자 이탈률은 40%에서 12%로 감소했고, 팀원들과의 협업을 통해 문제를 해결하는 과정에서 기술적 전문성과 팀워크의 중요성을 깊이 깨달았습니다. 이 경험은 네이버에서의 업무에도 큰 도움이 될 것입니다.
---
[ExperienceRole.PRIMARY] Cardify 프로젝트 - 결제 시스템 장애 해결
  사용할 사실: ['결제 API 응답 시간을 3초에서 200ms로 개선', '사용자 이탈률을 40%에서 12%로 감소', 'Eager Loading과 Redis 캐싱 도입']
  언급 방식: Cardify 프로젝트에서 결제 시스템의 장애를 해결하며 기술적 전문성과 팀워크의 중요성을 배웠습니다.


---
## 3단계: 초안 생성 (few-shot 예시 + 스토리라인 기반)

In [6]:
# === 문항 유형별 few-shot 예시 ===
# 직무 중립적으로 작성 - 구조와 흐름을 보여주는 것이 목적

FEWSHOT_EXAMPLES = {
    "지원동기": """[예시 - 지원동기 유형, 500자]
대학 3학년, 한 번의 서비스 장애가 수만 명의 일상을 멈출 수 있다는 사실을 체감했습니다. 직접 운영하던 캠퍼스 커뮤니티 앱에서 DB 커넥션 풀 고갈로 2시간 동안 접속 불가 상태가 발생했고, 기말고사 정보를 공유하던 3,000명의 사용자로부터 항의가 쏟아졌습니다. 장애 원인을 추적하며 밤새 코드를 뜯어본 그 경험이, 안정적인 서비스를 만들겠다는 다짐의 시작점이었습니다. 이후 커넥션 풀 모니터링과 Auto-scaling을 적용하여 동시접속 5,000명에서도 99.9% 가용성을 유지하는 구조로 개선했습니다. 이 과정에서 문제를 감추지 않고 투명하게 공유하는 장애 리포트 문화를 팀에 정착시켰고, 사후 분석이 반복 장애를 줄이는 가장 확실한 방법이라는 걸 몸으로 익혔습니다. 대규모 트래픽 환경에서 이런 경험을 확장하고 싶습니다.""",

    "직무역량": """[예시 - 직무역량 유형, 500자]
레거시 모놀리식 시스템의 배포에 매번 4시간이 걸렸습니다. 전체 빌드-테스트-배포 사이클을 돌려야 했고, 배포 중 장애가 발생하면 롤백에만 1시간이 소요되었습니다. 가장 먼저 병목 지점을 수치로 파악했습니다. APM 도구로 측정한 결과, 빌드 시간의 60%가 불필요한 통합 테스트에서 소비되고 있었습니다. 핵심 도메인 3개를 독립 서비스로 분리하고, 각 서비스별 CI/CD 파이프라인을 구축했습니다. 변경된 서비스만 빌드하도록 설계하니 배포 시간이 4시간에서 20분으로 줄었습니다. 카나리 배포를 도입하여 트래픽의 5%로 먼저 검증한 뒤 전체 반영하는 방식으로 전환했고, 배포 실패율이 월 3회에서 0회로 감소했습니다. 수치로 문제를 정의하고, 단계적으로 해결하는 접근법을 가장 잘 할 수 있습니다.""",

    "협업소통": """[예시 - 협업소통 유형, 500자]
기획자가 "빨리 만들어달라"고 했고, 개발팀은 "스펙이 불명확하다"고 했습니다. 서로 다른 언어를 쓰고 있었습니다. 프로젝트 3주 차에 일정이 2주 밀린 상황에서, 원인을 분석해보니 기획 문서에 적힌 요구사항과 개발팀이 이해한 요구사항 사이에 해석 차이가 15건 있었습니다. 매주 금요일 30분씩 "스펙 싱크" 미팅을 제안했습니다. 기획자가 화면을 그리고, 개발자가 그 자리에서 기술적 제약을 설명하는 방식이었습니다. 단순해 보이지만 효과는 확실했습니다. 해석 차이로 인한 재작업이 주 5건에서 0건으로 사라졌고, 밀려있던 일정을 1주 만에 정상 궤도로 복구했습니다. 결국 프로젝트를 당초 기한 내에 완료했습니다. 문제가 생겼을 때 누군가를 탓하기보다, 구조를 바꿔서 같은 문제가 반복되지 않게 만드는 것이 제가 팀에서 하는 역할입니다.""",

    "문제해결": """[예시 - 문제해결 유형, 500자]
서비스 오픈 3일 전, 부하 테스트에서 동시접속 1,000명 시점에 서버가 다운되었습니다. 목표는 5,000명이었습니다. 로그를 분석하니 특정 API에서 매 요청마다 무거운 집계 쿼리를 실행하고 있었습니다. 실시간 집계가 필요한 데이터가 아니었기 때문에, 1분 주기 배치로 전환하고 결과를 캐싱하는 방식으로 변경했습니다. 해당 API의 응답 시간이 2.3초에서 50ms로 감소했습니다. 그러나 다른 병목이 드러났습니다. 파일 업로드 API에서 동기 처리로 인한 스레드 블로킹이 발생하고 있었습니다. 비동기 처리와 S3 Presigned URL 방식으로 변경하여 서버 부하를 분산시켰습니다. 최종 부하 테스트에서 동시접속 6,000명까지 안정적으로 처리하는 것을 확인했고, 서비스는 예정대로 오픈했습니다. 이 경험으로 병목은 하나를 해결하면 다음 것이 드러난다는 걸 배웠습니다.""",

    "성장가치관": """[예시 - 성장가치관 유형, 500자]
첫 프로젝트에서 맡은 건 로그인 기능이었습니다. 간단하다고 생각했지만, 세션 관리, CSRF 방어, 비밀번호 해싱까지 고려하니 2주가 걸렸습니다. 코드 리뷰에서 선배가 남긴 코멘트 23개를 받았을 때, 솔직히 자존심이 상했습니다. 하지만 하나씩 수정하면서 "왜 이렇게 해야 하는지"를 직접 찾아보기 시작했습니다. OWASP Top 10 문서를 읽고, 실제 보안 취약점 사례를 분석했습니다. 3개월 후 두 번째 프로젝트에서는 인증 모듈 설계를 맡았고, 이번에는 코드 리뷰 코멘트가 3개로 줄었습니다. 지금은 팀 내에서 신규 입사자의 코드 리뷰를 담당하고 있습니다. 리뷰할 때 항상 "이유"를 함께 적습니다. 지적이 아니라 학습의 기회가 되어야 한다고 생각하기 때문입니다. 부족함을 인정하는 데서 성장이 시작된다는 것을, 그 첫 프로젝트에서 배웠습니다.""",

    "기타": "",  # 기타 유형은 예시 없이 진행
}

print(f"Few-shot 예시 {len(FEWSHOT_EXAMPLES)}개 유형 로드 완료")
print(f"현재 문항 유형: {classification.question_type}")
print(f"해당 예시 글자수: {len(FEWSHOT_EXAMPLES[classification.question_type.value])}자")

Few-shot 예시 6개 유형 로드 완료
현재 문항 유형: QuestionType.PROBLEM_SOLVING
해당 예시 글자수: 455자


In [ ]:
GENERATION_PROMPT = """당신은 자기소개서 작성 전문가입니다. 아래 스토리라인과 참고 예시를 바탕으로 자기소개서를 작성하세요.

## 지원 정보
- 회사: {company}
- 채용공고: {recruit_notice}

## 문항
{question}

## 글자수 제한
{max_length}자 (반드시 {min_length}~{max_length}자 사이로 작성)

## 스토리라인
- 핵심 메시지: {core_message}
- 도입: {opening}
- 전개: {development}
- 마무리: {conclusion}

## 경험별 사용 계획
{experience_plan}

## 참고 예시 (구조와 흐름만 참고, 내용을 복사하지 마세요)
{fewshot_example}

## 작성 규칙
1. 스토리라인의 흐름(도입→전개→마무리)을 따르세요.
2. [주 경험]은 구체적 사실과 수치를 포함하여 상세히 서술하세요.
3. [보조 경험]은 한 문장 이내로 자연스럽게 녹여 넣으세요.
4. 경험 정보에 없는 사실을 만들지 마세요.
5. '-했습니다' 체를 사용하세요.
6. '저는', '저의' 등 '저' 대명사를 사용하지 마세요.
7. 자기소개서 본문만 출력하세요. 부가 설명, 인사말, 제목을 넣지 마세요.

## 쉼표 사용 규칙 (매우 중요)
- '우선,', '또한,', '그리고,', '결과적으로,' 처럼 접속어 뒤에 쉼표를 쓰지 마세요.
- 한 문장 안에서 쉼표는 진짜 열거(A, B, C)가 아니면 쓰지 마세요.
- 쉼표 대신 문장을 끊어서 짧게 나누거나, 조사와 어미로 연결하세요.
  - 나쁜 예: "이를 해결하기 위해 A를 도입하고, B를 적용하여 C를 개선했습니다."
  - 좋은 예: "이를 해결하기 위해 A를 도입했습니다. B를 적용하여 C를 개선했습니다."

## 금지 표현 (절대 사용 금지)
다음 표현들은 AI 티가 나는 상투적 표현입니다. 절대 사용하지 마세요:
- "~할 수 있었습니다" → "~했습니다"로 직접 서술
- "~하는 계기가 되었습니다"
- "~의 중요성을 깨달았습니다" / "~을 깊이 깨달았습니다"
- "이 경험을 통해" / "이러한 경험을 통해"
- "이러한 ~을 통해" / "이를 통해"
- "결과적으로"
- "~에 큰 도움이 될 것입니다"
- "~을 몸소 느꼈습니다"
- "성장할 수 있었습니다" / "배울 수 있었습니다"
- "~하게 되었습니다" (피동형 약한 표현)"""


async def generate_draft(
    storyline: Storyline,
    question_type: QuestionType,
    question: str,
    max_length: int,
    company: str,
    recruit_notice: str,
) -> str:
    """스토리라인 기반 초안 생성"""
    # 경험별 사용 계획 텍스트
    experience_plan = "\n".join(
        f"- [{exp.role}] {exp.title}: {exp.how_to_mention}\n"
        f"  핵심 사실: {', '.join(exp.key_facts)}"
        for exp in storyline.experiences
    )

    # few-shot 예시
    fewshot = FEWSHOT_EXAMPLES.get(question_type.value, "")
    if not fewshot:
        fewshot = "참고 예시 없음"

    prompt = GENERATION_PROMPT.format(
        company=company,
        recruit_notice=recruit_notice,
        question=question,
        max_length=max_length,
        min_length=int(max_length * 0.9),
        core_message=storyline.core_message,
        opening=storyline.opening,
        development=storyline.development,
        conclusion=storyline.conclusion,
        experience_plan=experience_plan,
        fewshot_example=fewshot,
    )

    response = await llm.ainvoke(prompt)
    return response.content


# 테스트
draft = await generate_draft(
    storyline=storyline,
    question_type=classification.question_type,
    question=question,
    max_length=max_length,
    company=company,
    recruit_notice=recruit_notice,
)

print(f"=== 초안 ({len(draft)}자) ===")
print(draft)
print(f"\n글자수: {len(draft)}자 / 제한: {max_length}자")
print(f"글자수 준수: {'✅' if len(draft) <= max_length else '❌ 초과'}")

---
## 4단계: 검수 & 글자수 보정

In [8]:
class ReviewResult(BaseModel):
    """검수 결과"""
    has_listing_pattern: bool = Field(
        description="경험을 나열식으로 서술하고 있으면 True"
    )
    has_hallucination: bool = Field(
        description="경험 정보에 없는 사실이 포함되어 있으면 True"
    )
    feedback: str = Field(
        description="구체적인 개선 피드백 (문제가 없으면 빈 문자열)"
    )


REVIEW_PROMPT = """아래 자기소개서 초안을 검수하세요.

## 초안
{draft}

## 원본 경험 정보
{experiences}

## 검수 항목
1. **나열식 패턴**: 경험들이 "첫째~, 둘째~" 또는 별도 단락으로 나열되어 있는지 확인. 하나의 흐름으로 자연스럽게 연결되어야 합니다.
2. **할루시네이션**: 원본 경험 정보에 없는 수치, 성과, 역할, 활동이 포함되어 있는지 확인.

문제가 있으면 구체적으로 어떤 부분이 문제인지 피드백하세요."""


LENGTH_ADJUSTMENT_PROMPT = """아래 자기소개서의 글자수를 조정하세요.

## 현재 초안 ({current_length}자)
{draft}

## 목표
{min_length}~{max_length}자 사이로 조정하세요. (현재 {direction})

## 규칙
- 핵심 내용과 구체적 수치는 유지하세요.
- 자연스러운 문장 흐름을 유지하세요.
- 자기소개서 본문만 출력하세요."""


REVISION_PROMPT = """아래 자기소개서 초안을 피드백에 따라 수정하세요.

## 현재 초안
{draft}

## 피드백
{feedback}

## 원본 경험 정보
{experiences}

## 글자수 제한
{min_length}~{max_length}자

## 규칙
- 피드백의 지적사항을 반영하여 수정하세요.
- 경험 정보에 없는 사실을 만들지 마세요.
- 자기소개서 본문만 출력하세요."""


async def review_draft(
    draft: str,
    experiences_text: str,
) -> ReviewResult:
    """초안 검수"""
    structured_llm = classification_llm.with_structured_output(ReviewResult)
    result = await structured_llm.ainvoke(
        REVIEW_PROMPT.format(draft=draft, experiences=experiences_text)
    )
    return result


async def adjust_length(draft: str, max_length: int) -> str:
    """글자수 보정"""
    current = len(draft)
    min_length = int(max_length * 0.9)

    if min_length <= current <= max_length:
        return draft  # 범위 내면 그대로

    direction = "초과" if current > max_length else "부족"
    response = await llm.ainvoke(
        LENGTH_ADJUSTMENT_PROMPT.format(
            current_length=current,
            draft=draft,
            min_length=min_length,
            max_length=max_length,
            direction=f"{abs(current - max_length)}자 {direction}",
        )
    )
    return response.content


async def revise_draft(
    draft: str,
    feedback: str,
    experiences_text: str,
    max_length: int,
) -> str:
    """피드백 기반 수정"""
    response = await llm.ainvoke(
        REVISION_PROMPT.format(
            draft=draft,
            feedback=feedback,
            experiences=experiences_text,
            min_length=int(max_length * 0.9),
            max_length=max_length,
        )
    )
    return response.content


# === 검수 + 보정 파이프라인 실행 ===
experiences_text = assign_experience_roles(experiences_with_scores)

print("=== 검수 시작 ===")
review = await review_draft(draft, experiences_text)
print(f"나열식 패턴: {'⚠️ 있음' if review.has_listing_pattern else '✅ 없음'}")
print(f"할루시네이션: {'⚠️ 있음' if review.has_hallucination else '✅ 없음'}")
if review.feedback:
    print(f"피드백: {review.feedback}")

final_draft = draft

# 검수 문제가 있으면 수정
if review.has_listing_pattern or review.has_hallucination:
    print("\n=== 피드백 반영 수정 ===")
    final_draft = await revise_draft(final_draft, review.feedback, experiences_text, max_length)
    print(f"수정 후: {len(final_draft)}자")

# 글자수 보정 (최대 2회 시도)
min_length = int(max_length * 0.9)
for attempt in range(2):
    current = len(final_draft)
    if min_length <= current <= max_length:
        print(f"\n✅ 글자수 적합: {current}자")
        break
    print(f"\n=== 글자수 보정 (시도 {attempt + 1}) ===")
    print(f"현재: {current}자, 목표: {min_length}~{max_length}자")
    final_draft = await adjust_length(final_draft, max_length)
    print(f"보정 후: {len(final_draft)}자")

print(f"\n=== 최종 결과 ({len(final_draft)}자) ===")
print(final_draft)

=== 검수 시작 ===
나열식 패턴: ✅ 없음
할루시네이션: ✅ 없음
피드백: 자기소개서 초안은 전반적으로 잘 작성되어 있으며, 경험을 자연스럽게 연결하여 나열식 패턴이 없습니다. 각 경험이 하나의 흐름으로 잘 이어져 있습니다. 또한, 원본 경험 정보에 기반하여 작성되었기 때문에 할루시네이션 문제도 없습니다. 다만, 다음과 같은 개선점을 고려해볼 수 있습니다:

1. **구체성 강화**: 기술적 접근 방법에 대한 설명이 좋지만, 각 기술이 어떻게 문제 해결에 기여했는지에 대한 구체적인 예시를 추가하면 더 좋을 것 같습니다. 예를 들어, Eager Loading과 Redis 캐싱이 실제로 어떻게 성능을 개선했는지에 대한 간단한 설명이 추가되면 좋겠습니다.

2. **팀워크 강조**: 팀원들과의 협업을 강조하는 부분이 있지만, 구체적으로 어떤 역할을 맡았는지, 팀원들과 어떻게 소통했는지에 대한 예시를 추가하면 팀워크의 중요성을 더 잘 전달할 수 있을 것입니다.

3. **결과 강조**: 결과 부분에서 사용자 이탈률 감소와 API 응답 시간 단축을 강조했지만, 이 성과가 서비스에 미친 긍정적인 영향(예: 사용자 만족도 증가, 매출 증가 등)에 대한 언급이 추가되면 더 설득력 있는 내용이 될 것입니다.

✅ 글자수 적합: 457자

=== 최종 결과 (457자) ===
대학 동아리에서 진행한 Cardify 프로젝트는 결제 시스템의 장애로 인해 사용자 이탈률이 40%까지 증가한 위기 상황이었습니다. 이 문제를 해결하기 위해 팀원들과 함께 머리를 맞대고 원인을 분석했습니다. 우선, DB 쿼리 프로파일링을 통해 N+1 문제를 발견했습니다. 이를 해결하기 위해 Eager Loading과 Redis 캐싱을 도입하여 데이터 처리 속도를 개선했습니다. 또한, Circuit Breaker 패턴을 적용하여 외부 PG사 장애가 발생하더라도 서비스가 안정적으로 운영될 수 있도록 했습니다. 이러한 기술적 접근을 통해 API 응답 시간을 3초에서 200ms로 단축시킬 수 있었습니다. 결과

---
## 전체 파이프라인 통합 테스트

In [9]:
async def full_pipeline(
    user_message: str,
    question: str,
    max_length: int,
    company: str,
    recruit_notice: str,
    experiences_with_scores: list[dict],
):
    """전체 파이프라인 실행"""
    print("=" * 60)
    print("[1/4] 요청 분류")
    print("=" * 60)
    classification = await classify_request(user_message, question)
    print(f"자기소개서 요청: {classification.is_cover_letter_request}")
    print(f"문항 유형: {classification.question_type}")

    if not classification.is_cover_letter_request:
        print("\n→ 자기소개서 요청이 아님 → 일반 대화로 처리")
        return None

    print(f"\n{'=' * 60}")
    print("[2/4] 스토리라인 설계")
    print("=" * 60)
    storyline = await design_storyline(
        question, max_length, company, experiences_with_scores
    )
    print(f"핵심 메시지: {storyline.core_message}")
    print(f"도입: {storyline.opening}")
    print(f"전개: {storyline.development}")
    print(f"마무리: {storyline.conclusion}")
    for exp in storyline.experiences:
        print(f"  [{exp.role}] {exp.title}: {exp.how_to_mention}")

    print(f"\n{'=' * 60}")
    print("[3/4] 초안 생성")
    print("=" * 60)
    draft = await generate_draft(
        storyline, classification.question_type,
        question, max_length, company, recruit_notice,
    )
    print(f"초안 ({len(draft)}자):")
    print(draft)

    print(f"\n{'=' * 60}")
    print("[4/4] 검수 & 보정")
    print("=" * 60)
    experiences_text = assign_experience_roles(experiences_with_scores)

    review = await review_draft(draft, experiences_text)
    print(f"나열식: {'⚠️' if review.has_listing_pattern else '✅'}")
    print(f"할루시네이션: {'⚠️' if review.has_hallucination else '✅'}")

    final_draft = draft

    if review.has_listing_pattern or review.has_hallucination:
        print(f"피드백: {review.feedback}")
        final_draft = await revise_draft(
            final_draft, review.feedback, experiences_text, max_length
        )

    min_length = int(max_length * 0.9)
    for attempt in range(2):
        current = len(final_draft)
        if min_length <= current <= max_length:
            break
        print(f"글자수 보정 (시도 {attempt + 1}): {current}자 → 목표 {min_length}~{max_length}자")
        final_draft = await adjust_length(final_draft, max_length)

    print(f"\n{'=' * 60}")
    print(f"최종 결과 ({len(final_draft)}자 / 제한 {max_length}자)")
    print("=" * 60)
    print(final_draft)

    return final_draft


# 실행
result = await full_pipeline(
    user_message=user_message,
    question=question,
    max_length=max_length,
    company=company,
    recruit_notice=recruit_notice,
    experiences_with_scores=experiences_with_scores,
)

[1/4] 요청 분류
자기소개서 요청: True
문항 유형: QuestionType.PROBLEM_SOLVING

[2/4] 스토리라인 설계
핵심 메시지: 문제를 해결하는 과정에서 기술적 전문성과 팀워크의 중요성을 배웠습니다.
도입: 대학 동아리에서 진행한 Cardify 프로젝트는 결제 시스템의 장애로 인해 사용자 이탈률이 급증하는 위기를 맞았습니다. 평균 3초에 달하는 결제 API 응답 시간은 40%의 사용자 이탈을 초래했습니다.
전개: 이 문제를 해결하기 위해, 저는 DB 쿼리 프로파일링을 통해 N+1 문제를 발견했습니다. 이를 해결하기 위해 Eager Loading과 Redis 캐싱을 도입하고, Circuit Breaker 패턴을 적용하여 외부 PG사 장애 전파를 차단했습니다. 이러한 기술적 접근을 통해 API 응답 시간을 3초에서 200ms로 93% 개선할 수 있었습니다.
마무리: 결과적으로 사용자 이탈률은 40%에서 12%로 감소했습니다. 이 경험을 통해 문제 해결 과정에서의 기술적 전문성과 팀원 간의 협력의 중요성을 깊이 깨달았습니다. 또한, 해커톤에서의 실시간 채팅 서비스 개발 경험은 저의 기술적 역량을 더욱 확장하는 계기가 되었습니다.
  [ExperienceRole.PRIMARY] Cardify 프로젝트 - 결제 시스템 장애 해결: Cardify 프로젝트에서 결제 시스템의 장애를 해결하며 기술적 전문성을 발휘했습니다.

[3/4] 초안 생성
초안 (445자):
대학 동아리에서 진행한 Cardify 프로젝트는 결제 시스템의 장애로 사용자 이탈률이 급증하는 위기를 맞았습니다. 평균 3초에 달하는 결제 API 응답 시간은 40%의 사용자 이탈을 초래했습니다. 이 문제를 해결하기 위해 DB 쿼리 프로파일링을 통해 N+1 문제를 발견했습니다. 이를 해결하기 위해 Eager Loading과 Redis 캐싱을 도입하고, Circuit Breaker 패턴을 적용하여 외부 PG사 장애 전파를 차단했습니다. 이러한 기술적 접근을 통해 API 응답 시간을 3초에서

---
## 비교 테스트: 기존 방식 vs 새 파이프라인

In [ ]:
import importlib.util, os

spec = importlib.util.spec_from_file_location(
    "prompts",
    os.path.join(project_root, "src/chats/prompts.py")
)
prompts_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prompts_module)
COVER_LETTER_SYSTEM_PROMPT = prompts_module.COVER_LETTER_SYSTEM_PROMPT


async def old_pipeline(user_message, question, max_length, company, recruit_notice, experiences_with_scores):
    """기존 방식으로 생성 (비교용)"""
    exp_text = ""
    for exp in experiences_with_scores:
        exp_text += f"### {exp['title']}\n"
        exp_text += f"- 유형: {exp['experience_type']}\n"
        if exp.get('format_type') == 'STAR':
            exp_text += f"- 상황(S): {exp.get('situation', '')}\n"
            exp_text += f"- 과제(T): {exp.get('task', '')}\n"
            exp_text += f"- 행동(A): {exp.get('action', '')}\n"
            exp_text += f"- 결과(R): {exp.get('result', '')}\n"
        elif exp.get('format_type') == 'PSI':
            exp_text += f"- 문제(P): {exp.get('problem', '')}\n"
            exp_text += f"- 해결(S): {exp.get('solution', '')}\n"
            exp_text += f"- 인사이트(I): {exp.get('insight', '')}\n"
        exp_text += f"- 카테고리: {exp['category']}\n\n"

    prompt = COVER_LETTER_SYSTEM_PROMPT.format(
        company=company,
        recruit_notice=recruit_notice,
        question=question,
        max_length=max_length,
        experiences=exp_text,
        allowed_facts="제한 없음",
    )

    response = await llm.ainvoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": user_message},
    ])
    return response.content


# 기존 방식
print("=== 기존 방식 ===")
old_result = await old_pipeline(
    user_message, question, max_length, company, recruit_notice, experiences_with_scores
)
print(f"({len(old_result)}자)")
print(old_result)

print(f"\n\n{'=' * 60}\n\n")

# 새 파이프라인
print("=== 새 파이프라인 ===")
print(f"({len(result)}자)")
print(result)